# Multi-hop RAG with Question Decomposition

## Project goal

Implement and inspect a multi-hop Retrieval-Augmented Generation (RAG) pipeline on a 30-question subset of HotpotQA. This notebook reuses the corpus, embeddings, and FAISS index created in Notebook 01.

## Pipeline overview

1. Decompose the original question into a small set of searchable subquestions.
2. Embed each subquestion and retrieve its top-*k* chunks from the existing FAISS index.
3. Deduplicate retrieved chunks.
4. Provide the original question and retrieved evidence to Qwen2.5-7B-Instruct to generate a concise answer.
5. Evaluate the answer with normalized Exact Match and inspect representative errors.

## Experimental setup

- **Dataset:** HotpotQA; 30 evaluation questions
- **Generator:** Qwen2.5-7B-Instruct, 8-bit quantized
- **Embedding model:** `sentence-transformers/all-MiniLM-L6-v2`
- **Vector search:** FAISS `IndexFlatIP`
- **Retrieval setting:** top 5 chunks per subquestion, with duplicate chunks removed

> **Scope note:** This notebook implements the decomposition-based pipeline and explores answer-generation behavior. It does not yet establish that question decomposition improves over direct RAG.


## 1. Environment Setup

Install the required libraries, mount Google Drive, and define the project paths.

In [ ]:

%pip install -q transformers accelerate bitsandbytes sentence-transformers faiss-cpu


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 78.4 MB/s eta 0:00:00


In [ ]:
from pathlib import Path
from google.colab import drive

# Mount Google Drive
drive.mount("/content/drive")

# Define the project root directory
PROJECT_ROOT = Path("/content/drive/MyDrive/Multi_hop_RAG")

# Define project directories
DATA_DIR = PROJECT_ROOT / "data"
CORPUS_DIR = PROJECT_ROOT / "corpus"
INDEX_DIR = PROJECT_ROOT / "indexes"
RESULTS_DIR = PROJECT_ROOT / "results"

# Create the results directory if it does not exist
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Display project paths
print(f"Project root: {PROJECT_ROOT}")
print(f"Data directory: {DATA_DIR}")
print(f"Corpus directory: {CORPUS_DIR}")
print(f"Index directory: {INDEX_DIR}")
print(f"Results directory: {RESULTS_DIR}")

# Check whether the main project directory exists
assert PROJECT_ROOT.exists(), (
    f"Project directory not found: {PROJECT_ROOT}"
)

print("\nProject directory verified successfully.")


Mounted at /content/drive
Project root: /content/drive/MyDrive/Multi_hop_RAG
Data directory: /content/drive/MyDrive/Multi_hop_RAG/data
Corpus directory: /content/drive/MyDrive/Multi_hop_RAG/corpus
Index directory: /content/drive/MyDrive/Multi_hop_RAG/indexes
Results directory: /content/drive/MyDrive/Multi_hop_RAG/results

Project directory verified successfully.


## 2. Load the Language Model



In [ ]:
import sys

MODEL_MODULE_DIR = Path("/content/drive/MyDrive")

if str(MODEL_MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODEL_MODULE_DIR))

# Import the existing LLM interface
from qwen_model import call_llm

print("LLM interface imported successfully.")


Loading Qwen tokenizer...
Model path: /content/drive/MyDrive/Model_qwen2.5-7B-instruct/models--Qwen--Qwen2.5-7B-Instruct/snapshots/a09a35458c702b33eeacc393d103063234e8bc28


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading Qwen2.5-7B-Instruct in 8-bit...


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Qwen model loaded successfully.
Device: cuda:0
VRAM allocated: 8.71 GB
LLM interface imported successfully.


## 3. Load embedding model

Model: `sentence-transformers/all-MiniLM-L6-v2`

In [ ]:
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME
)

print("✓ Embedding model loaded.")
print(f"Model: {EMBEDDING_MODEL_NAME}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✓ Embedding model loaded.
Model: sentence-transformers/all-MiniLM-L6-v2


## 4. Load the Corpus and FAISS Index

Load the existing corpus and vector index from Google Drive. The assertion checks that the number of corpus chunks matches the number of indexed vectors.



In [ ]:

import json
import faiss
import os
import numpy as np

DATASET_NAME = "hotpotqa"
N_SAMPLES = 7405
N_EVAL = 30

# Define file paths
corpus_path = f"{PROJECT_ROOT}/corpus/{DATASET_NAME}/corpus.json"
index_path = f"{PROJECT_ROOT}/indexes/{DATASET_NAME}/faiss.index"

# Load corpus
assert os.path.exists(corpus_path), (
    f"Corpus file not found: {corpus_path}"
)

with open(corpus_path, "r", encoding="utf-8") as f:
    corpus = json.load(f)

print(f"Corpus loaded successfully: {len(corpus)} chunks")

# Load FAISS index
assert os.path.exists(index_path), (
    f"FAISS index file not found: {index_path}"
)

index = faiss.read_index(index_path)

print("FAISS index loaded successfully.")
print(f"Index type       : {type(index).__name__}")
print(f"Embedding dim    : {index.d}")
print(f"Number of vectors: {index.ntotal}")

# Verify corpus and index consistency
assert len(corpus) == index.ntotal, (
    f"Corpus/index mismatch: {len(corpus)} chunks "
    f"but {index.ntotal} indexed vectors"
)

print("\nCorpus and FAISS index are ready.")


Corpus loaded successfully: 73700 chunks
FAISS index loaded successfully.
Index type       : IndexFlatIP
Embedding dim    : 384
Number of vectors: 73700

Corpus and FAISS index are ready.


## 5. Load the Evaluation Set

Load the fixed set of 30 HotpotQA evaluation examples. Keeping this set fixed makes it possible to compare later experiments on the same questions.


In [ ]:
# Define the evaluation file path
evaluation_path = f"{DATA_DIR}/{DATASET_NAME}/evaluation_30.json"

# Verify that the evaluation file exists
assert os.path.exists(evaluation_path), (
    f"Evaluation file not found: {evaluation_path}"
)

# Load evaluation samples
with open(evaluation_path, "r", encoding="utf-8") as f:
    evaluation_samples = json.load(f)

print(f"Evaluation samples loaded: {len(evaluation_samples)}")

# Verify the evaluation dataset
assert len(evaluation_samples) == N_EVAL, (
    f"Expected {N_EVAL} evaluation samples, "
    f"but found {len(evaluation_samples)}"
)

# Display one example
print("\nExample evaluation question:")
print(evaluation_samples[0]["question"])
print("\nExpected answer:")
print(evaluation_samples[0]["answer"])


Evaluation samples loaded: 30

Example evaluation question:
What was Iqbal F. Qadir on when he participated in an attack on a radar station located on western shore of the Okhamandal Peninsula?

Expected answer:
flotilla


## 6. Decompose the Question into Independent Subquestions

The LLM is prompted to produce up to three focused, searchable subquestions before retrieval begins. This is **parallel/independent decomposition**: the system generates the subquestions in one step rather than using the answer to one subquestion to formulate the next.

The function returns an empty list if the response cannot be parsed as the expected JSON or if the model returns the original question unchanged. Some pipeline cells then fall back to the original question.

In [ ]:
def decompose_question(question, max_subquestions=4):
    """
    Decompose a multi-hop question into independent subquestions.

    Args:
        question (str): The original question.
        max_subquestions (int): Maximum number of subquestions.

    Returns:
        list[str]: A list of independent subquestions.
    """


    prompt = f"""
    You are a question decomposition assistant for a multi-hop question answering system.

    Your task is to break down the original question into a small number of simpler subquestions that help retrieve the information needed to answer it.

    Follow these rules:
    - Identify exactly what information the original question asks for.
    - Preserve the meaning of the original question.
    - Generate up to 3 simple subquestions that help find the
      requested answer.
    - Each subquestion must seek a specific piece of information
      needed to answer the original question.
    - Do not ask about details that are already stated in the
      original question.
    - Do not ask about related topics that do not help determine
      the requested answer.
    - If the question can be answered directly, return the original
      question as the only subquestion.
    - Do not answer the questions.
    - Return valid JSON only, using this format:

    {{"subquestions": ["Question 1", "Question 2"]}}

    Original question:
    {question}
    """


    response = call_llm(prompt)

    # Remove Markdown code fences if present
    response = response.strip()

    if response.startswith("```"):
        lines = response.splitlines()

        if lines and lines[0].startswith("```"):
            lines = lines[1:]

        if lines and lines[-1].strip() == "```":
            lines = lines[:-1]

        response = "\n".join(lines).strip()

    # Parse the JSON response
    try:
        parsed = json.loads(response)
        subquestions = parsed.get("subquestions", [])
    except (json.JSONDecodeError, TypeError, AttributeError):
        print("Warning: Failed to parse model response as JSON.")
        print("Raw response:", repr(response))
        return []

    # Keep valid, non-empty question strings
    subquestions = [
        item.strip()
        for item in subquestions
        if isinstance(item, str) and item.strip()
    ]

    # Limit the number of subquestions
    subquestions = subquestions[:max_subquestions]

    # Detect an unchanged copy of the original question
    if (
        len(subquestions) == 1
        and subquestions[0].rstrip("?.!").lower()
        == question.strip().rstrip("?.!").lower()
    ):
        print("Warning: The model returned the original question unchanged.")
        return []

    return subquestions


### Smoke test: inspect one decomposition

This test illustrates the format of the generated subquestions. A plausible decomposition is not, by itself, evidence that retrieval or final-answer accuracy has improved.

In [ ]:
test_question = evaluation_samples[0]["question"]

subquestions = decompose_question(test_question)

print("Original question:")
print(test_question)

print("\nGenerated subquestions:")
for i, subquestion in enumerate(subquestions, start=1):
    print(f"{i}. {subquestion}")

Original question:
What was Iqbal F. Qadir on when he participated in an attack on a radar station located on western shore of the Okhamandal Peninsula?

Generated subquestions:
1. What attack did Iqbal F. Qadir participate in?
2. Where was the radar station located?
3. On what specific occasion did this occur?


## 7. Retrieve Documents for Each Subquestion

Each subquestion is embedded separately and used to retrieve its top 5 chunks from FAISS. Retrieved chunks are merged in retrieval order and deduplicated by `chunk_id`.

The final number of unique chunks can be smaller than the total number of retrieval results because different subquestions may retrieve the same chunk.

In [ ]:
# Retrieve documents for each subquestion

TOP_K = 5

retrieved_documents = []
seen_chunk_ids = set()

for subquestion in subquestions:
    # Encode the subquestion
    query_embedding = embedding_model.encode(
        [subquestion],
        normalize_embeddings=True
    ).astype("float32")

    # Retrieve the top-k chunks
    scores, indices = index.search(
        query_embedding,
        TOP_K
    )

    # Collect retrieved chunks and remove duplicates
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue

        chunk = corpus[idx]
        chunk_id = chunk["chunk_id"]

        if chunk_id in seen_chunk_ids:
            continue

        seen_chunk_ids.add(chunk_id)

        retrieved_documents.append({
            "chunk_id": chunk_id,
            "sample_id": chunk.get("sample_id"),
            "title": chunk.get("title"),
            "text": chunk["text"],
            "score": float(score),
            "retrieved_for": subquestion
        })

print(f"Number of subquestions: {len(subquestions)}")
print(f"Number of unique retrieved chunks: {len(retrieved_documents)}")


Number of subquestions: 3
Number of unique retrieved chunks: 15


In [ ]:
# Display retrieved documents

for i, document in enumerate(retrieved_documents, start=1):
    print(f"\n--- Retrieved Document {i} ---")
    print("Chunk ID:", document["chunk_id"])
    print("Title:", document["title"])
    print("Score:", round(document["score"], 4))
    print("Retrieved for:", document["retrieved_for"])
    print("Text:", document["text"][:500])



--- Retrieved Document 1 ---
Chunk ID: 5a7a567255429941d65f25bd_chunk_9
Title: Iqbal F. Qadir
Score: 0.545
Retrieved for: What attack did Iqbal F. Qadir participate in?
Text: Vice-Admiral Iqbal Fazl Quadir (Urdu:اقبال فضل قادر) , is a retired three-star rank admiral in the Pakistan Navy, former diplomat, and a defence analyst.  He is renown for his participation in second war with India when he was part of the flotilla that attacked the radar station in Dwarka, India.

--- Retrieved Document 2 ---
Chunk ID: 5abda2b755429933744ab80f_chunk_3
Title: Cecil Chaudhry
Score: 0.5073
Retrieved for: What attack did Iqbal F. Qadir participate in?
Text: Group Captain Cecil Chaudhry (27 August 1941 – 13 April 2012) was a Pakistani academic, human rights activist, and veteran fighter pilot.  As a Flight Lieutenant, he fought in the Indo-Pakistani war of 1965 and later, as a Squadron Leader, in the Indo-Pakistani War of 1971.  During the 1965 war, Chaudhry and three other pilots, under the leadershi


## 8. Generate the Final Answer

The original question and all retrieved evidence are combined into one prompt. The model is instructed to answer directly from the evidence, avoid guessing, and return only the answer rather than an explanation.

In [ ]:

def generate_answer(question, retrieved_documents):
    """Generate an answer using the retrieved evidence."""

    # Combine retrieved documents into a single context
    evidence_context = "\n\n".join(
        [
            f"Document {i}\n"
            f"Title: {doc['title']}\n"
            f"Content: {doc['text']}"
            for i, doc in enumerate(retrieved_documents, start=1)
        ]
    )

    # Build the prompt
    prompt = f"""
Answer the original question using the provided evidence.

Instructions:
- Identify exactly what the question asks for.
- Find the phrase in the evidence that directly answers the question.
- Prefer an explicit phrase from the evidence over an inference.
- Do not reinterpret the question as asking for a different detail.
- Do not say the evidence is insufficient if a direct answer is present.
- Return only the answer, without explanations.
- Do not guess or use outside knowledge.


Question:
{question}

Evidence:
{evidence_context}

Answer:
"""

    # Generate the answer
    answer = call_llm(
        prompt,
        max_tokens=128
    )

    return answer.strip()


In [ ]:
# Generate and display the final answer

final_answer = generate_answer(
    test_question,
    retrieved_documents
)

print("Original question:")
print(test_question)

print("\nGenerated answer:")
print(final_answer)


Original question:
What was Iqbal F. Qadir on when he participated in an attack on a radar station located on western shore of the Okhamandal Peninsula?

Generated answer:
flotilla


## 9. Run a Small Debugging Experiment

Inspect three examples end to end: the original question, generated subquestions, retrieved documents, and final answer. This is a qualitative debugging step, not a separate accuracy estimate.


In [ ]:

# Run a small debugging experiment

N_DEBUG = 3

for sample_idx, sample in enumerate(evaluation_samples[:N_DEBUG], start=1):
    question = sample["question"]
    gold_answer = sample["answer"]

    print("\n" + "=" * 80)
    print(f"DEBUG EXAMPLE {sample_idx}")
    print("=" * 80)

    # 1. Decompose the original question
    subquestions = decompose_question(question)

    # Fall back to the original question if decomposition fails
    if not subquestions:
        subquestions = [question]

    print("\nOriginal question:")
    print(question)

    print("\nGenerated subquestions:")
    for i, subquestion in enumerate(subquestions, start=1):
        print(f"{i}. {subquestion}")

    # 2. Retrieve documents for each subquestion
    retrieved_documents = []
    seen_chunk_ids = set()

    for subquestion in subquestions:
        query_embedding = embedding_model.encode(
            [subquestion],
            normalize_embeddings=True
        ).astype("float32")

        scores, indices = index.search(
            query_embedding,
            TOP_K
        )

        for score, idx in zip(scores[0], indices[0]):
            if idx == -1:
                continue

            chunk = corpus[idx]
            chunk_id = chunk["chunk_id"]

            if chunk_id in seen_chunk_ids:
                continue

            seen_chunk_ids.add(chunk_id)

            retrieved_documents.append({
                "chunk_id": chunk_id,
                "sample_id": chunk.get("sample_id"),
                "title": chunk.get("title"),
                "text": chunk["text"],
                "score": float(score),
                "retrieved_for": subquestion
            })

    print("\nNumber of unique retrieved documents:",
          len(retrieved_documents))

    # 3. Inspect the first five retrieved documents
    print("\nRetrieved evidence (up to 5 documents):")

    for i, doc in enumerate(retrieved_documents[:5], start=1):
        print(f"\nDocument {i}")
        print("Title:", doc["title"])
        print("Score:", round(doc["score"], 4))
        print("Retrieved for:", doc["retrieved_for"])
        print("Text:", doc["text"][:500])

    # 4. Generate the final answer
    final_answer = generate_answer(
        question,
        retrieved_documents
    )

    print("\nFinal answer:")
    print(final_answer)

    print("\nGold answer:")
    print(gold_answer)



DEBUG EXAMPLE 1

Original question:
What was Iqbal F. Qadir on when he participated in an attack on a radar station located on western shore of the Okhamandal Peninsula?

Generated subquestions:
1. What attack did Iqbal F. Qadir participate in?
2. Where was the radar station located?
3. On what specific date or during what time period did this attack occur?

Number of unique retrieved documents: 15

Retrieved evidence (up to 5 documents):

Document 1
Title: Iqbal F. Qadir
Score: 0.545
Retrieved for: What attack did Iqbal F. Qadir participate in?
Text: Vice-Admiral Iqbal Fazl Quadir (Urdu:اقبال فضل قادر) , is a retired three-star rank admiral in the Pakistan Navy, former diplomat, and a defence analyst.  He is renown for his participation in second war with India when he was part of the flotilla that attacked the radar station in Dwarka, India.

Document 2
Title: Cecil Chaudhry
Score: 0.5073
Retrieved for: What attack did Iqbal F. Qadir participate in?
Text: Group Captain Cecil Chaudhr

## 10. Evaluate on the 30-Question Test Set

Run the complete decomposition-based pipeline on the fixed evaluation subset. The evaluation records Exact Match, the number of retrieved documents, timing, and call counts.

**Evaluation note:** normalized Exact Match is strict. A response that contains the correct answer but adds extra text may still be marked incorrect.

In [ ]:
import re
import time
import pandas as pd
from IPython.display import display


def normalize_answer(answer):
    """Normalize an answer for simple Exact Match evaluation."""
    answer = str(answer).lower().strip()
    answer = re.sub(r"[^\w\s]", "", answer)
    answer = re.sub(r"\s+", " ", answer)
    return answer


results = []

N_EVAL = min(30, len(evaluation_samples))
total_llm_calls = 0
total_retrieval_calls = 0

for sample_idx, sample in enumerate(evaluation_samples[:N_EVAL], start=1):
    question = sample["question"]
    gold_answer = sample["answer"]

    total_start = time.perf_counter()

    # Step 1: Decompose the question
    subquestions = decompose_question(question)
    total_llm_calls += 1

    if not subquestions:
        subquestions = [question]

    # Step 2: Retrieve documents for each subquestion
    retrieved_documents = []
    seen_chunk_ids = set()

    for subquestion in subquestions:
        query_embedding = embedding_model.encode(
            [subquestion],
            normalize_embeddings=True
        ).astype("float32")

        total_retrieval_calls += 1

        scores, indices = index.search(query_embedding, TOP_K)

        for score, idx in zip(scores[0], indices[0]):
            if idx < 0:
                continue

            chunk = corpus[idx]

            if chunk["chunk_id"] in seen_chunk_ids:
                continue

            seen_chunk_ids.add(chunk["chunk_id"])

            retrieved_documents.append({
                "chunk_id": chunk["chunk_id"],
                "sample_id": chunk["sample_id"],
                "title": chunk["title"],
                "text": chunk["text"],
                "score": float(score),
                "retrieved_for": subquestion
            })

    # Step 3: Generate the final answer
    generation_start = time.perf_counter()

    total_llm_calls += 1
    generated_answer = generate_answer(
        question,
        retrieved_documents
    )

    generation_time = time.perf_counter() - generation_start
    total_time = time.perf_counter() - total_start

    # Step 4: Calculate Exact Match
    exact_match = int(
        normalize_answer(generated_answer)
        == normalize_answer(gold_answer)
    )

    results.append({
        "Question": question,
        "Gold Answer": gold_answer,
        "Generated Answer": generated_answer,
        "Retrieved Docs": len(retrieved_documents),
        "Exact Match": exact_match,
        "Generation Time (s)": round(generation_time, 2),
        "Total Time (s)": round(total_time, 2),
        "LLM Calls": 2,
        "Retrieval Calls": len(subquestions),
    })

    print(f"Completed {sample_idx}/{N_EVAL}")


# Create the evaluation DataFrame
results_df = pd.DataFrame(results)

print("\nEvaluation Results:")
display(results_df)


Completed 1/30
Completed 2/30
Completed 3/30
Completed 4/30
Completed 5/30
Completed 6/30
Completed 7/30
Completed 8/30
Completed 9/30
Completed 10/30
Completed 11/30
Completed 12/30
Completed 13/30
Completed 14/30
Completed 15/30
Completed 16/30
Completed 17/30
Completed 18/30
Completed 19/30
Completed 20/30
Completed 21/30
Completed 22/30
Completed 23/30
Completed 24/30
Completed 25/30
Completed 26/30
Completed 27/30
Completed 28/30
Completed 29/30
Completed 30/30

Evaluation Results:


,Question,Gold Answer,Generated Answer,Retrieved Docs,Exact Match,Generation Time (s),Total Time (s),LLM Calls,Retrieval Calls
0,What was Iqbal F. Qadir on when he participate...,flotilla,Vice-Admiral Iqbal Fazl Quadir was part of the...,10,0,7.62,12.93,3,2
1,When did the park at which Tivolis Koncertsal ...,15 August 1843,1843,10,0,2.13,7.88,3,2
2,What is the shared country of ancestry between...,Armenian,Armenian,10,1,1.72,7.38,3,2
3,The school in which the Wilmslow Show is held ...,Centre of Excellence,good,10,0,1.49,7.98,3,2
4,Who will Billy Howle be seen opposite in the u...,Saoirse Ronan,Saoirse Ronan,10,1,1.96,9.06,3,2
5,"What animated movie, starring Danny Devito, fe...",The Lorax,Fire and Ice (1983 film),10,0,3.13,8.48,3,2
6,Out of the actors who have played the role of ...,Scott Adkins,Scott Adkins,20,1,6.37,19.90,5,4
7,Tommy's Honour was a drama film that included ...,War & Peace,"Document 1 Content: Jack Lowden, who played Yo...",7,0,6.68,14.60,3,2
8,Which rock band chose its name by drawing it o...,Midnight Oil,Midnight Oil,10,1,2.10,9.40,3,2
9,"""Tunak"", is a bhangra/pop love song by an arti...",1967,1969,5,0,1.41,5.66,2,1


In [ ]:
RESULTS_DIR = f"{PROJECT_ROOT}/results"
os.makedirs(RESULTS_DIR, exist_ok=True)

RESULTS_PATH = f"{RESULTS_DIR}/{DATASET_NAME}/rag_evaluation_results_30samples.csv"

results_df.to_csv(
    RESULTS_PATH,
    index=False,
    encoding="utf-8-sig"
)

print(f"Evaluation results saved to: {RESULTS_PATH}")

Evaluation results saved to: /content/drive/MyDrive/Multi_hop_RAG/results/hotpotqa/rag_evaluation_results_30samples.csv


## 11. Summary and Quality Checks

Review the aggregate metrics before inspecting individual errors. These values describe this particular run; because answer generation is sampled, a later run may produce different answers.

In [ ]:
# Summary metrics
accuracy = results_df["Exact Match"].mean() * 100
avg_retrieved_docs = results_df["Retrieved Docs"].mean()
avg_generation_time = results_df["Generation Time (s)"].mean()
avg_total_time = results_df["Total Time (s)"].mean()

print(f"\nNumber of evaluated questions: {len(results_df)}")
print(f"Exact Match Accuracy: {accuracy:.2f}%")
print(f"Average Retrieved Documents: {avg_retrieved_docs:.2f}")
print(f"Average Generation Time: {avg_generation_time:.2f} seconds")
print(f"Average Total Time per Question: {avg_total_time:.2f} seconds")

print(f"\nTotal LLM Calls: {total_llm_calls}")
print(f"Total Retrieval Calls: {total_retrieval_calls}")


print(
    "Average LLM Calls per Question:",
    results_df["LLM Calls"].mean()
)

print(
    f"Average Retrieval Calls per Question: "
    f"{results_df['Retrieval Calls'].mean():.2f}"
)


Number of evaluated questions: 30
Exact Match Accuracy: 30.00%
Average Retrieved Documents: 9.00
Average Generation Time: 2.86 seconds
Average Total Time per Question: 9.44 seconds

Total LLM Calls: 60
Total Retrieval Calls: 57
Average LLM Calls per Question: 2.0
Average Retrieval Calls per Question: 1.90


## 12. Inspect Incorrect Answers

First display the examples that failed Exact Match. The following cells then inspect selected cases to understand whether the issue may involve missing evidence, answer extraction, or answer formatting.

### 12.1. First-pass error inspection

This table lists answers that did not match the reference answer under the notebook's normalized Exact Match metric. It is an initial error inventory; it does not automatically identify the underlying cause.

In [ ]:
wrong_answers = results_df[
    results_df["Exact Match"] == 0
][[
    "Question",
    "Gold Answer",
    "Generated Answer"
]]

display(wrong_answers)

,Question,Gold Answer,Generated Answer
0,What was Iqbal F. Qadir on when he participate...,flotilla,Vice-Admiral Iqbal Fazl Quadir was part of the...
1,When did the park at which Tivolis Koncertsal ...,15 August 1843,1843
3,The school in which the Wilmslow Show is held ...,Centre of Excellence,good
5,"What animated movie, starring Danny Devito, fe...",The Lorax,Fire and Ice (1983 film)
7,Tommy's Honour was a drama film that included ...,War & Peace,"Document 1 Content: Jack Lowden, who played Yo..."
9,"""Tunak"", is a bhangra/pop love song by an arti...",1967,1969
10,Which Captain launched the attack which led to...,Captain John Underhill,Willem Kieft
12,What was the concept of the business Eric S .P...,to ensure wide visibility and understanding of...,The evidence does not provide information abou...
13,Which port city lies approximately 25 km north...,Keelung,Hong Kong
14,Has Stefan Edberg won more events than Édouar...,yes,No


### 12.2. Trace selected examples through the pipeline

For these selected questions, inspect the generated subquestions and retrieved evidence alongside the gold and generated answers. Since this cell reruns decomposition and retrieval, it makes additional LLM calls.

In [ ]:
DEBUG_INDICES = [0, 1, 3]

for idx in DEBUG_INDICES:
    sample = evaluation_samples[idx]

    question = sample["question"]
    gold_answer = sample["answer"]

    print("\n" + "=" * 80)
    print(f"Sample index: {idx}")
    print("\nQuestion:")
    print(question)
    print("\nGold answer:")
    print(gold_answer)

    subquestions = decompose_question(question)

    if not subquestions:
        subquestions = [question]

    print("\nSubquestions:")
    for i, subquestion in enumerate(subquestions, start=1):
        print(f"{i}. {subquestion}")

    retrieved_documents = []
    seen_chunk_ids = set()

    for subquestion in subquestions:
        query_embedding = embedding_model.encode(
            [subquestion],
            normalize_embeddings=True
        ).astype("float32")

        scores, indices = index.search(query_embedding, TOP_K)

        for score, doc_idx in zip(scores[0], indices[0]):
            if doc_idx < 0:
                continue

            chunk = corpus[doc_idx]

            if chunk["chunk_id"] in seen_chunk_ids:
                continue

            seen_chunk_ids.add(chunk["chunk_id"])

            retrieved_documents.append({
                "chunk_id": chunk["chunk_id"],
                "sample_id": chunk["sample_id"],
                "title": chunk["title"],
                "text": chunk["text"],
                "score": float(score),
                "retrieved_for": subquestion
            })

    print("\nRetrieved documents:")

    for i, doc in enumerate(retrieved_documents, start=1):
        print(f"\nDocument {i}")
        print(f"Title: {doc['title']}")
        print(f"Score: {doc['score']:.4f}")
        print(f"Retrieved for: {doc['retrieved_for']}")
        print(f"Text: {doc['text'][:700]}")


Sample index: 0

Question:
What was Iqbal F. Qadir on when he participated in an attack on a radar station located on western shore of the Okhamandal Peninsula?

Gold answer:
flotilla

Subquestions:
1. What was Iqbal F. Qadir's role during the attack on the radar station?
2. Where was the radar station located?
3. On which date did the attack on the radar station occur?

Retrieved documents:

Document 1
Title: Iqbal F. Qadir
Score: 0.5831
Retrieved for: What was Iqbal F. Qadir's role during the attack on the radar station?
Text: Vice-Admiral Iqbal Fazl Quadir (Urdu:اقبال فضل قادر) , is a retired three-star rank admiral in the Pakistan Navy, former diplomat, and a defence analyst.  He is renown for his participation in second war with India when he was part of the flotilla that attacked the radar station in Dwarka, India.

Document 2
Title: Cecil Chaudhry
Score: 0.5391
Retrieved for: What was Iqbal F. Qadir's role during the attack on the radar station?
Text: Group Captain Cecil Chaudh

### 12.3. Sanity check with manually focused evidence

These short tests provide the generator with concise evidence that directly states the answer. They test whether the model can extract an answer under favorable conditions; they do not measure end-to-end RAG performance.

In [ ]:
test_cases = [
    {
        "question": (
            "What was Iqbal F. Qadir on when he "
            "participated in the attack?"
        ),
        "evidence": (
            "He was part of the flotilla that attacked "
            "the radar station in Dwarka, India."
        ),
        "gold_answer": "flotilla"
    },
    {
        "question": "When did Tivoli Gardens open?",
        "evidence": (
            "The park opened on 15 August 1843 and is "
            "the second-oldest operating amusement park."
        ),
        "gold_answer": "15 August 1843"
    }
]

for case in test_cases:
    prompt = f"""
Answer the question using only the evidence.

Return only the shortest complete answer directly
supported by the evidence. Do not return a full sentence.

Question:
{case["question"]}

Evidence:
{case["evidence"]}

Answer:
"""

    answer = call_llm(prompt, max_tokens=32)

    print("\nQuestion:", case["question"])
    print("Gold answer:", case["gold_answer"])
    print("Generated answer:", answer.strip())


Question: What was Iqbal F. Qadir on when he participated in the attack?
Gold answer: flotilla
Generated answer: flotilla

Question: When did Tivoli Gardens open?
Gold answer: 15 August 1843
Generated answer: 15 August 1843


### Exploratory test: answer generation with fixed evidence

The following experiments focus on a small number of questions and reuse retrieved evidence where possible. They help inspect answer-generation behavior, but they are not a replacement for a controlled evaluation over the full test set.


### 12.4. Re-test answer generation on selected evaluation questions

This cell revisits a few questions and compares the earlier output with a fresh generation using retrieved evidence. The result can vary because the LLM uses sampling.

In [ ]:
# Re-test Answer Generation

TEST_INDICES = [0, 1, 3]

for idx in TEST_INDICES:
    row = results_df.iloc[idx]

    print("\n" + "=" * 70)
    print(f"Sample index: {idx}")
    print("Question:", row["Question"])
    print("Gold answer:", row["Gold Answer"])
    print("Previous answer:", row["Generated Answer"])

    # Retrieve the documents for this question again
    question = row["Question"]
    subquestions = decompose_question(question)

    retrieved_documents = []
    seen_chunk_ids = set()

    for subquestion in subquestions:
        query_embedding = embedding_model.encode(
            [subquestion],
            normalize_embeddings=True
        ).astype("float32")

        scores, indices = index.search(
            query_embedding, TOP_K
        )

        for score, doc_idx in zip(scores[0], indices[0]):
            if doc_idx < 0:
                continue

            doc = corpus[doc_idx]
            chunk_id = doc["chunk_id"]

            if chunk_id not in seen_chunk_ids:
                seen_chunk_ids.add(chunk_id)

                retrieved_documents.append({
                    "chunk_id": chunk_id,
                    "sample_id": doc["sample_id"],
                    "title": doc["title"],
                    "text": doc["text"],
                    "score": float(score),
                    "retrieved_for": subquestion
                })

    new_answer = generate_answer(
        question, retrieved_documents
    )

    print("New answer:", new_answer)
    print("Retrieved documents:", len(retrieved_documents))



Sample index: 0
Question: What was Iqbal F. Qadir on when he participated in an attack on a radar station located on western shore of the Okhamandal Peninsula?
Gold answer: flotilla
Previous answer: Vice-Admiral Iqbal Fazl Quadir was part of the flotilla that attacked the radar station in Dwarka, India.
New answer: Vice-Admiral Iqbal Fazl Quadir participated in the attack on the radar station in Dwarka, India.
Retrieved documents: 14

Sample index: 1
Question: When did the park at which Tivolis Koncertsal is located open?
Gold answer: 15 August 1843
Previous answer: 1843
New answer: 1843
Retrieved documents: 10

Sample index: 3
Question: The school in which the Wilmslow Show is held is designated as what?
Gold answer: Centre of Excellence
Previous answer: good
New answer: Centre of Excellence
Retrieved documents: 10


### 12.5. Inspect the evidence for the Tivoli Gardens question

This experiment examines the documents retrieved for a question whose reference answer is a full date. It helps separate evidence availability from answer-generation behavior.

In [ ]:

# Prepare fixed evidence for a controlled generation test

idx = 1
question = results_df.iloc[idx]["Question"]

subquestions = decompose_question(question)

fixed_documents = []
seen_chunk_ids = set()

for subquestion in subquestions:
    query_embedding = embedding_model.encode(
        [subquestion],
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = index.search(query_embedding, TOP_K)

    for score, doc_idx in zip(scores[0], indices[0]):
        if doc_idx < 0:
            continue

        doc = corpus[doc_idx]
        chunk_id = doc["chunk_id"]

        if chunk_id not in seen_chunk_ids:
            seen_chunk_ids.add(chunk_id)
            fixed_documents.append({
                "chunk_id": chunk_id,
                "sample_id": doc["sample_id"],
                "title": doc["title"],
                "text": doc["text"],
                "score": float(score),
                "retrieved_for": subquestion
            })

print("Question:", question)
print("\nSubquestions:")
for sq in subquestions:
    print("-", sq)

print("\nRetrieved evidence:")
for i, doc in enumerate(fixed_documents, 1):
    print(f"\nDocument {i}: {doc['title']}")
    print(doc["text"])


Question: When did the park at which Tivolis Koncertsal is located open?

Subquestions:
- What is the name of the park where Tivolis Koncertsal is located?
- When did this park open?

Retrieved evidence:

Document 1: Tivolis Koncertsal
Tivolis Koncertsal is a 1,660-capacity concert hall located at Tivoli Gardens in Copenhagen, Denmark.  The building, which was designed by Frits Schlegel and Hans Hansen, was built between 1954 and 1956.

Document 2: Tivolis Koncertsal
Tivolis Koncertsal is a 1,660-capacity concert hall located at Tivoli Gardens in Copenhagen, Denmark.  The building, which was designed by Frits Schlegel and Hans Hansen, was built between 1954 and 1956.

Document 3: Tivoli Gardens
Tivoli Gardens (or simply Tivoli) is a famous amusement park and pleasure garden in Copenhagen, Denmark.  The park opened on 15 August 1843 and is the second-oldest operating amusement park in the world, after Dyrehavsbakken in nearby Klampenborg, also in Denmark.

Document 4: Benjakitti Park
Be

### 12.6. Repeated generation with the same evidence

Generate multiple answers from the same question and fixed documents. Differences between runs indicate that output sampling can affect the result even when the evidence is unchanged.

In [ ]:
# Test answer generation with fixed retrieved evidence

question = results_df.iloc[1]["Question"]

print("Question:", question)
print("Gold answer:", results_df.iloc[1]["Gold Answer"])

for run in range(1, 4):
    answer = generate_answer(question, fixed_documents)

    print(f"\nRun {run}:")
    print(answer)


Question: When did the park at which Tivolis Koncertsal is located open?
Gold answer: 15 August 1843

Run 1:
1843

Run 2:
1843

Run 3:
1843


### 12.7. Compare two prompt formulations on one example

This is a small, single-question prompt experiment. It can suggest a direction for further testing, but it cannot establish a general improvement by itself.

In [ ]:
# Compare the current prompt with a stricter answer prompt

question = results_df.iloc[1]["Question"]

# Test 1: Existing generation function
current_answer = generate_answer(question, fixed_documents)

print("Test 1 - Current generate_answer:")
print(current_answer)


# Test 2: Explicit answer extraction prompt
evidence = "\n\n".join(
    f"Title: {doc['title']}\n{doc['text']}"
    for doc in fixed_documents
)

strict_prompt = f"""
Answer the question using only the evidence below.

Question:
{question}

Evidence:
{evidence}

Instructions:
- Return the complete answer, not just part of it.
- If the answer is a date, include the day, month, and year
  whenever all three are available in the evidence.
- Do not add explanations or a full sentence.
- Return only the answer.

Answer:
"""

strict_answer = call_llm(strict_prompt, max_tokens=32)

print("\nTest 2 - Explicit answer prompt:")
print(strict_answer.strip())


Test 1 - Current generate_answer:
1843

Test 2 - Explicit answer prompt:
15 August 1843


### 12.8. Define an alternative answer-generation prompt

The function below is an experimental alternative to `generate_answer`. It is retained for comparison and does not replace the original function in the main pipeline.

In [ ]:
# New Answer Generation Function

def new_generate_answer(question, retrieved_documents):
    evidence = "\n\n".join(
        f"Title: {doc['title']}\n{doc['text']}"
        for doc in retrieved_documents
    )

    prompt = f"""
Answer the question using only the evidence below.

Question:
{question}

Evidence:
{evidence}

Instructions:
- Answer the original question directly using only the evidence.
- Return the shortest complete answer that fully answers the question.
- Preserve all essential details, such as dates, names, locations,
  and multi-word titles or descriptions, when needed to answer the question.
- Do not return a partial answer if the evidence provides a more
  complete answer.
- Do not include explanations or unrelated details.
- Do not invent information that is not supported by the evidence.

Answer:
"""

    answer = call_llm(prompt, max_tokens=128)
    return answer.strip()


# Test on selected questions using the existing evaluation results
TEST_INDICES = [0, 1, 3]

for idx in TEST_INDICES:
    row = results_df.iloc[idx]
    question = row["Question"]

    # Reuse the same retrieved documents if available for this question.
    # For index 1, fixed_documents contains the evidence from our previous test.
    if idx == 1:
        docs = fixed_documents
    else:
        subquestions = decompose_question(question)
        docs = []
        seen_chunk_ids = set()

        for subquestion in subquestions:
            query_embedding = embedding_model.encode(
                [subquestion],
                normalize_embeddings=True
            ).astype("float32")

            scores, indices = index.search(query_embedding, TOP_K)

            for score, doc_idx in zip(scores[0], indices[0]):
                if doc_idx < 0:
                    continue

                doc = corpus[doc_idx]
                chunk_id = doc["chunk_id"]

                if chunk_id not in seen_chunk_ids:
                    seen_chunk_ids.add(chunk_id)
                    docs.append({
                        "chunk_id": chunk_id,
                        "sample_id": doc["sample_id"],
                        "title": doc["title"],
                        "text": doc["text"],
                        "score": float(score),
                        "retrieved_for": subquestion
                    })

    new_answer = new_generate_answer(question, docs)

    print("\n" + "=" * 70)
    print(f"Sample index: {idx}")
    print("Question:", question)
    print("Gold answer:", row["Gold Answer"])
    print("Old answer:", row["Generated Answer"])
    print("New answer:", new_answer)



Sample index: 0
Question: What was Iqbal F. Qadir on when he participated in an attack on a radar station located on western shore of the Okhamandal Peninsula?
Gold answer: flotilla
Old answer: Vice-Admiral Iqbal Fazl Quadir was part of the flotilla that attacked the radar station in Dwarka, India.
New answer: Iqbal F. Qadir participated in an attack on a radar station located in Dwarka, India, during the second war between India and Pakistan.

Sample index: 1
Question: When did the park at which Tivolis Koncertsal is located open?
Gold answer: 15 August 1843
Old answer: 1843
New answer: 15 August 1843

Sample index: 3
Question: The school in which the Wilmslow Show is held is designated as what?
Gold answer: Centre of Excellence
Old answer: good
New answer: good


### 12.9. Test answer extraction with focused versus distracting evidence

These manually constructed examples explore how the evidence wording affects the generated answer. They are diagnostic examples, not a benchmark evaluation.

In [ ]:
# Focused Evidence Test

focused_tests = [
    {
        "index": 0,
        "question": (
            "What was Iqbal F. Qadir on when he participated "
            "in an attack on a radar station located on "
            "western shore of the Okhamandal Peninsula?"
        ),
        "gold_answer": "flotilla",
        "focused_evidence": (
            "Iqbal F. Qadir was part of the flotilla that "
            "attacked the radar station in Dwarka, India."
        ),
        "distracting_evidence": (
            "Vice-Admiral Iqbal Fazl Quadir was part of the "
            "flotilla that attacked the radar station in Dwarka, India. "
            "The attack occurred during the second war between "
            "India and Pakistan."
        )
    },
    {
        "index": 3,
        "question": (
            "The school in which the Wilmslow Show is held "
            "is designated as what?"
        ),
        "gold_answer": "Centre of Excellence",
        "focused_evidence": (
            "Wilmslow High School, where the Wilmslow Show is held, "
            "is designated as a Centre of Excellence."
        ),
        "distracting_evidence": (
            "The Wilmslow Show is held at Wilmslow High School. "
            "The school is a designated Centre of Excellence. "
            "The school is also designated 'good' by Ofsted."
        )
    }
]

def run_focused_test(question, evidence):
    prompt = f"""
Answer the question using only the evidence.

Question:
{question}

Evidence:
{evidence}

Return only the shortest complete answer that directly answers
the question. Do not add explanations.

Answer:
"""
    return call_llm(prompt, max_tokens=48).strip()


for test in focused_tests:
    print("\n" + "=" * 70)
    print("Question:", test["question"])
    print("Gold answer:", test["gold_answer"])

    answer_focused = run_focused_test(
        test["question"],
        test["focused_evidence"]
    )

    answer_with_distractor = run_focused_test(
        test["question"],
        test["distracting_evidence"]
    )

    print("\nAnswer with focused evidence:")
    print(answer_focused)

    print("\nAnswer with additional information:")
    print(answer_with_distractor)



Question: What was Iqbal F. Qadir on when he participated in an attack on a radar station located on western shore of the Okhamandal Peninsula?
Gold answer: flotilla

Answer with focused evidence:
radar station in Dwarka, India

Answer with additional information:
Vice-Admiral Iqbal Fazl Quadir was on a flotilla.

Question: The school in which the Wilmslow Show is held is designated as what?
Gold answer: Centre of Excellence

Answer with focused evidence:
Centre of Excellence

Answer with additional information:
Centre of Excellence


## 13. Compare the Old and New Answer Prompts on the Same 30 Questions

For each question, both generation prompts receive the same retrieved documents. This isolates the prompt formulation more effectively than comparing separate runs with different retrieved evidence. However, the experiment still uses sampled generation and only one run, so the result should be treated as preliminary.


In [ ]:
# Compare Old vs New Answer Generation on the Same 30 Questions

import pandas as pd
import time
import re

N_COMPARE = min(30, len(evaluation_samples))
comparison_results = []

def normalize_answer(text):
    """Normalize answers for a simple Exact Match comparison."""
    text = str(text).lower().strip()
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text

def retrieve_documents_for_question(question):
    """Decompose a question and retrieve deduplicated documents."""
    subquestions = decompose_question(question)
    documents = []
    seen_chunk_ids = set()

    for subquestion in subquestions:
        query_embedding = embedding_model.encode(
            [subquestion],
            normalize_embeddings=True
        ).astype("float32")

        scores, indices = index.search(query_embedding, TOP_K)

        for score, doc_idx in zip(scores[0], indices[0]):
            if doc_idx < 0:
                continue

            doc = corpus[doc_idx]
            chunk_id = doc["chunk_id"]

            if chunk_id not in seen_chunk_ids:
                seen_chunk_ids.add(chunk_id)
                documents.append({
                    "chunk_id": chunk_id,
                    "sample_id": doc["sample_id"],
                    "title": doc["title"],
                    "text": doc["text"],
                    "score": float(score),
                    "retrieved_for": subquestion
                })

    return documents


for i in range(N_COMPARE):
    sample = evaluation_samples[i]

    # Adapt these keys if your evaluation samples use different names.
    question = sample.get("question", sample.get("Question", ""))
    gold_answer = sample.get("answer", sample.get("Gold Answer", ""))

    if not question or not gold_answer:
        print(f"Skipping sample {i}: question or answer not found.")
        continue

    documents = retrieve_documents_for_question(question)

    start = time.time()
    old_answer = generate_answer(question, documents)
    old_time = time.time() - start

    start = time.time()
    new_answer = new_generate_answer(question, documents)
    new_time = time.time() - start

    gold_norm = normalize_answer(gold_answer)
    old_em = int(normalize_answer(old_answer) == gold_norm)
    new_em = int(normalize_answer(new_answer) == gold_norm)

    comparison_results.append({
        "Index": i,
        "Question": question,
        "Gold Answer": gold_answer,
        "Old Answer": old_answer,
        "New Answer": new_answer,
        "Old EM": old_em,
        "New EM": new_em,
        "Old Time (s)": round(old_time, 2),
        "New Time (s)": round(new_time, 2),
        "Retrieved Docs": len(documents)
    })

    print(f"\n[{i + 1}/{N_COMPARE}]")
    print("Gold:", gold_answer)
    print("Old:", old_answer, "| EM:", old_em)
    print("New:", new_answer, "| EM:", new_em)


comparison_df = pd.DataFrame(comparison_results)

if not comparison_df.empty:
    old_accuracy = comparison_df["Old EM"].mean() * 100
    new_accuracy = comparison_df["New EM"].mean() * 100

    print("\n" + "=" * 60)
    print("COMPARISON SUMMARY")
    print("=" * 60)
    print(f"Questions evaluated: {len(comparison_df)}")
    print(f"Old Exact Match Accuracy: {old_accuracy:.2f}%")
    print(f"New Exact Match Accuracy: {new_accuracy:.2f}%")
    print(f"Accuracy change: {new_accuracy - old_accuracy:+.2f} percentage points")

    improved = comparison_df[
        (comparison_df["Old EM"] == 0) &
        (comparison_df["New EM"] == 1)
    ]
    regressed = comparison_df[
        (comparison_df["Old EM"] == 1) &
        (comparison_df["New EM"] == 0)
    ]

    print("Wrong → Correct:", len(improved))
    print("Correct → Wrong:", len(regressed))

    display(comparison_df[
        (comparison_df["Old EM"] != comparison_df["New EM"])
    ])

    output_path = (
        f"{PROJECT_ROOT}/results/{DATASET_NAME}/"
        "old_vs_new_answer_generation_30samples.csv"
    )
    comparison_df.to_csv(output_path, index=False)
    print("\nSaved results to:", output_path)



[1/30]
Gold: flotilla
Old: radar station in Dwarka, India | EM: 0
New: Iqbal F. Qadir participated in an attack on the radar station in Dwarka, India, during the second war with India. | EM: 0

[2/30]
Gold: 15 August 1843
Old: 1843 | EM: 0
New: 15 August 1843 | EM: 1

[3/30]
Gold: Armenian
Old: Armenian | EM: 1
New: Armenian | EM: 1

[4/30]
Gold: Centre of Excellence
Old: good | EM: 0
New: good | EM: 0

[5/30]
Gold: Saoirse Ronan
Old: Saoirse Ronan | EM: 1
New: Saoirse Ronan | EM: 1

[6/30]
Gold: The Lorax
Old: Fire and Ice (1983 film) | EM: 0
New: None of the provided evidence mentions an animated movie starring Danny Devito with music written and produced by Kool Kojak. | EM: 0

[7/30]
Gold: Scott Adkins
Old: Jean-Claude Van Damme | EM: 0
New: Jean-Claude Van Damme | EM: 0

[8/30]
Gold: War & Peace
Old: Document 1
Content: The film is directed by Jason Connery, and the father and son are portrayed by Peter Mullan and Jack Lowden. | EM: 0
New: Jack Lowden found success with the 2016 

,Index,Question,Gold Answer,Old Answer,New Answer,Old EM,New EM,Old Time (s),New Time (s),Retrieved Docs
1,1,When did the park at which Tivolis Koncertsal ...,15 August 1843,1843,15 August 1843,0,1,2.25,2.98,10
8,8,Which rock band chose its name by drawing it o...,Midnight Oil,Midnight Oil,Midnight Oil chose its name by drawing it out ...,1,0,2.10,3.87,10



Saved results to: /content/drive/MyDrive/Multi_hop_RAG/results/hotpotqa/old_vs_new_answer_generation_30samples.csv


## 14. Heuristic Error Categorization

The following rules assign preliminary labels based on the answer text and Exact Match outcomes. The categories are heuristic and can overlap conceptually; they should not be interpreted as verified root causes. In particular, this analysis cannot establish whether a failure originated in retrieval without examining the relevant retrieved evidence.

In [ ]:
# Error Analysis

import pandas as pd
import re

def classify_error(row):
    gold = str(row["Gold Answer"]).strip()
    old = str(row["Old Answer"]).strip()
    new = str(row["New Answer"]).strip()

    old_em = int(row["Old EM"])
    new_em = int(row["New EM"])

    old_norm = normalize_answer(old)
    new_norm = normalize_answer(new)
    gold_norm = normalize_answer(gold)

    # Did the model return a non-answer or an uncertainty statement?
    uncertainty_phrases = [
        "does not provide information",
        "does not contain information",
        "cannot be answered",
        "insufficient information",
        "no direct answer",
        "not enough information",
        "none of the provided evidence"
    ]

    if any(p in new.lower() for p in uncertainty_phrases):
        return "Possible unsupported / insufficient-evidence response"

    # Exact match was achieved
    if new_em == 1:
        return "Correct answer"

    # The new answer contains the gold answer, but adds extra text
    if gold_norm and gold_norm in new_norm:
        return "Correct answer with extra text"

    # Very short answers may omit essential details
    if len(new.split()) < len(gold.split()):
        return "Possible incomplete answer"

    # The answer changed from correct to incorrect
    if old_em == 1 and new_em == 0:
        return "Regression: previously correct answer became incorrect"

    # The answer changed but is still incorrect
    if old_norm != new_norm:
        return "Answer changed but remains incorrect"

    # The answer did not change and is incorrect
    return "Same incorrect answer"


analysis_df = comparison_df.copy()
analysis_df["Error Category"] = analysis_df.apply(
    classify_error, axis=1
)

print("Error category counts:")
display(
    analysis_df["Error Category"]
    .value_counts()
    .rename_axis("Category")
    .reset_index(name="Count")
)

print("\nIncorrect answers with categories:")
display(
    analysis_df.loc[
        analysis_df["New EM"] == 0,
        [
            "Index",
            "Question",
            "Gold Answer",
            "Old Answer",
            "New Answer",
            "Error Category"
        ]
    ].reset_index(drop=True)
)

analysis_path = (
    f"{PROJECT_ROOT}/results/{DATASET_NAME}/"
    "error_analysis_30_questions.csv"
)
analysis_df.to_csv(analysis_path, index=False)

print("\nSaved error analysis to:", analysis_path)


Error category counts:


,Category,Count
0,Answer changed but remains incorrect,10
1,Correct answer,8
2,Same incorrect answer,5
3,Possible unsupported / insufficient-evidence r...,3
4,Possible incomplete answer,2
5,Correct answer with extra text,2



Incorrect answers with categories:


,Index,Question,Gold Answer,Old Answer,New Answer,Error Category
0,0,What was Iqbal F. Qadir on when he participate...,flotilla,"radar station in Dwarka, India",Iqbal F. Qadir participated in an attack on th...,Answer changed but remains incorrect
1,3,The school in which the Wilmslow Show is held ...,Centre of Excellence,good,good,Possible incomplete answer
2,5,"What animated movie, starring Danny Devito, fe...",The Lorax,Fire and Ice (1983 film),None of the provided evidence mentions an anim...,Possible unsupported / insufficient-evidence r...
3,6,Out of the actors who have played the role of ...,Scott Adkins,Jean-Claude Van Damme,Jean-Claude Van Damme,Same incorrect answer
4,7,Tommy's Honour was a drama film that included ...,War & Peace,Document 1\nContent: The film is directed by J...,Jack Lowden found success with the 2016 BBC mi...,Answer changed but remains incorrect
5,8,Which rock band chose its name by drawing it o...,Midnight Oil,Midnight Oil,Midnight Oil chose its name by drawing it out ...,Correct answer with extra text
6,9,"""Tunak"", is a bhangra/pop love song by an arti...",1967,1969,1969,Same incorrect answer
7,10,Which Captain launched the attack which led to...,Captain John Underhill,The question cannot be answered with the given...,The evidence provided does not contain informa...,Possible unsupported / insufficient-evidence r...
8,11,In which film did Emilio Estevez star in in th...,The Outsiders,The Outsurers,The film in which Emilio Estevez starred in th...,Correct answer with extra text
9,12,What was the concept of the business Eric S .P...,to ensure wide visibility and understanding of...,The evidence provided does not contain any inf...,The evidence does not provide information abou...,Possible unsupported / insufficient-evidence r...



Saved error analysis to: /content/drive/MyDrive/Multi_hop_RAG/results/hotpotqa/error_analysis_30_questions.csv


## 15. Findings, Limitations, and Next Steps

### Main evaluation result

The recorded run on 30 questions produced the following results:

| Metric | Recorded result |
|---|---:|
| Exact Match accuracy | 30.00% (9/30) |
| Average retrieved documents per question | 9.00 |
| Average generation time | 2.86 seconds |
| Average total time per question | 9.44 seconds |
| Total LLM calls | 60 (2.00 per question) |
| Total retrieval calls | 57 (1.90 per question) |



### Answer-prompt comparison

In the separate 30-question comparison, the old and new answer-generation prompts both achieved **26.67% Exact Match**. The new prompt corrected one previously incorrect answer and made one previously correct answer incorrect. Therefore, this experiment did not show an overall accuracy improvement from the new prompt.

### What can and cannot be concluded

- The decomposition-based pipeline has been implemented and evaluated on a small fixed subset.
- The experiments show that answer generation can fail even when useful evidence is present, and that missing or irrelevant evidence can also cause failures.
- The model uses sampled generation, so individual answers and metrics may vary between runs.
- Normalized Exact Match penalizes answers that include extra explanatory text, even when the reference answer appears inside the response.

